This file imports a CSV file generated using the Excel Matrix format and 2-0 LCA's CSVmaker

# SimaProCSV importer

## 1. Import packages

In [4]:
import bw2data as bd
import bw2io as bi
import pandas as pd
from functions.utils import Config
from pathlib import Path

## 2. Set project and see databases
The databases are required here, because the Excel database we want to import in this script needs to be matched to these.

In [ ]:
bd.projects.set_current(Config.PROJECT_NAME)

In [2]:
# define a project where we install the databases and work in this script
bd.projects.set_current(Config.PROJECT_NAME)

In [3]:
bd.databases

Databases dictionary with 7 object(s):
	BONSAI v2.4.1
	BONSAI v2.4.1 biosphere
	bafu-2026
	biosphere3
	brightway_project_template
	ecoinvent-3.12-biosphere
	ecoinvent-3.12-consequential

## 3. Import the CSV file containing the model 
Import the CSV file 

In [ ]:
csv_path = Path("your_path") # Replace "your_path" with the actual path to your CSV file
assert csv_path.is_file()


In [ ]:
assert Config.BONSAI_DATABASE_NAME in bd.databases
assert Config.BONSAI_BIOSPHERE_NAME in bd.databases

importer = bi.SimaProBlockCSVImporter(
    path_or_stream=csv_path,
    database_name="DEA_capserv",
    biosphere_database_name=Config.BONSAI_BIOSPHERE_NAME,
    separate_products=False,
)


importer.apply_strategies()

2026-10-02 14:05:15.622 | INFO     | bw_simapro_csv.main:__init__:119 - Writing logs to C:\Users\TimWeber\AppData\Local\pylca\bw_simapro_csv\Logs\1.-2026-10-02T14-05-15
2026-10-02 14:05:15.640 | WARNING  | bw_simapro_csv.header:parse_header:146 - 
    Export is missing the SimaPro version line (e.g. `{SimaPro 9.5.0.2}`).
    Using default value of `None`
        
2026-10-02 14:05:15.644 | WARNING  | bw_simapro_csv.header:parse_header:154 - 
    Export is missing type (processes, methods, or product stages).
    Using default value of 'processes'
        
2026-10-02 14:05:15.650 | INFO     | bw_simapro_csv.main:__init__:142 - Using database name 'DEA_capserv'
2026-10-02 14:05:15.652 | INFO     | bw_simapro_csv.main:__init__:147 - SimaPro CSV import started.
	File: 'C:\Users\TimWeber\OneDrive - 2.-0 LCA Consultants ApS\Desktop\CSV_maker\DEA_csv_04082025_final\1..CSV'
	Delimiter: ';'
	Name: '(Not given)'
2026-10-02 14:05:15.694 | INFO     | bw_simapro_csv.main:resolve_parameters:324 - Ext

Applying strategy: set_metadata_using_single_functional_exchange
Applying strategy: override_process_name_using_single_functional_exchange
Applying strategy: drop_unspecified_subcategories
Applying strategy: split_simapro_name_geo
Applying strategy: create_products_as_new_nodes
Applying strategy: link_technosphere_based_on_name_unit_location
Applying strategy: link_iterable_by_fields
Applying strategy: match_internal_simapro_simapro_with_unit_conversion
Applied 8 strategies in 0.97 seconds


In [9]:
# Apply the appropriate importer.randonneur(...) mappings here.
# Their choice depends on the SimaPro export and target biosphere.

importer.normalize_labels_to_brightway_standard()

importer.match_database(
    Config.BONSAI_BIOSPHERE_NAME,
    fields=["name", "categories", "unit"],
    edge_kinds=["biosphere"],
)

importer.match_database(
    Config.BONSAI_DATABASE_NAME,
    fields=["name", "location", "unit", "activity type"],
    edge_kinds=["technosphere"],
)

importer.statistics()
display(pd.DataFrame(list(importer.unlinked)))

Applying strategy: normalize_simapro_labels_to_brightway_standard
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Graph statistics for `DEA_capserv` importer:
10 graph nodes:
	process: 10
697 graph edges:
	technosphere: 687
	production: 10
10 edges to the following databases:
	DEA_capserv: 10
351 unique unlinked edges (687 total):
	technosphere: 351




,name,unit,comment,line_no,amount,uncertainty type,loc,type,simapro_category
0,"38 treatment of waste/scrap, wood {CN}",ton,,67,7.253545e-07,0,7.253545e-07,technosphere,Materials/fuels
1,"38 treatment of waste/scrap, steel {CN}",ton,,68,2.144080e-02,0,2.144080e-02,technosphere,Materials/fuels
2,"38 treatment of waste/scrap, sewage {CN}",ton,,69,1.138037e-05,0,1.138037e-05,technosphere,Materials/fuels
3,"38 treatment of waste/scrap, plastic {CN}",ton,,70,8.184307e-04,0,8.184307e-04,technosphere,Materials/fuels
4,"38 treatment of waste/scrap, paper {CN}",ton,,71,1.413152e-04,0,1.413152e-04,technosphere,Materials/fuels
...,...,...,...,...,...,...,...,...,...
346,41 construction work {SE},MEUR2011,,1341,5.249328e-04,0,5.249328e-04,technosphere,Materials/fuels
347,62 computer and related services {SE},MEUR2011,,1342,1.894589e-04,0,1.894589e-04,technosphere,Materials/fuels
348,08.91 chemical and fertilizer minerals {SE},ton,,1343,8.632909e-06,0,8.632909e-06,technosphere,Materials/fuels
349,20.59 chemicals nec {SE},ton,,1344,0.000000e+00,0,0.000000e+00,technosphere,Materials/fuels


In [ ]:
assert importer.all_linked, "Resolve the remaining unlinked exchanges first."
importer.write_database()